# PART 1 – PYTHON & DATA FOUNDATIONS

## Task 1.1 – Read the File the Foundations Way

Objective: Read the original appointments.csv using Python's csv module, validate numeric data types, separate accepted and rejected records, and save rejected records as JSON.

In [2]:
import csv
import json

file_path = "data/appointments.csv"

with open(file_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    headers = reader.fieldnames
    rows = list(reader)

print("Headers:", headers)
print("Total rows:", len(rows))

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total rows: 1020


In [3]:
#creating parse row function

def parse_row(row):

    converted_row = row.copy()

    converted_row["age"] = int(row["age"])
    converted_row["fee_inr"] = int(row["fee_inr"])
    converted_row["prior_no_shows"] = int(row["prior_no_shows"])
    converted_row["distance_km"] = float(row["distance_km"])

    return converted_row

In [5]:
accepted = []
rejected = []

for row in rows:

    try:
        converted_row = parse_row(row)
        accepted.append(converted_row)

    except ValueError:
        rejected.append(row)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

Accepted rows: 988
Rejected rows: 32


In [6]:
#save rejected rows to new json file

with open("rejected_rows.json", "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

print("rejected_rows.json created successfully!")

rejected_rows.json created successfully!


## Task 1.2 – Inspect and Clean

Objective: Inspect the original appointments dataset and perform initial cleaning by removing duplicates, standardizing booking channels, converting dates, and handling missing appointment dates and statuses.

In [7]:
#loading dataset using pandas

import pandas as pd

df = pd.read_csv("data/appointments.csv")

print("Dataset Shape:", df.shape)

df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Dataset Shape: (1020, 14)
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

Missing Values:
appointment_id      0
patient_id          0
clinic_id           

In [8]:
# Step: Check distinct booking channel spellings in raw data

# Reload the original CSV without applying cleaning
raw_df = pd.read_csv("data/appointments.csv")

# Count distinct raw booking channel values
print(
    "Raw booking channel spellings:",
    raw_df["booking_channel"].nunique()
)

# Display all raw spellings
print(sorted(raw_df["booking_channel"].dropna().unique()))

Raw booking channel spellings: 16
[' App', ' phone', ' walk-in', 'APP ', 'App', 'App ', 'PHONE', 'Phone', 'Phone ', 'WALK-IN', 'Walk-In', 'Walk-in', 'Walk-in ', 'app', 'phone', 'walk-in']


In [9]:
#remove duplicates

df = df.drop_duplicates()

print("Rows after removing duplicates:", len(df))

Rows after removing duplicates: 1000


In [10]:
#cleaning booking_channel

df["booking_channel"] = df["booking_channel"].str.strip().str.title()

print("Unique booking channels:", df["booking_channel"].nunique())
print(df["booking_channel"].unique())
print("Rows after cleaning booking_channel:", len(df))

Unique booking channels: 3
<StringArray>
['Phone', 'Walk-In', 'App']
Length: 3, dtype: str
Rows after cleaning booking_channel: 1000


In [11]:
#date conversion

df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

print("Rows after date conversion:", len(df))

Rows after date conversion: 1000


In [12]:
#remove missing appointment dates and statuses

df = df.dropna(subset=["appt_date", "status"])

print("Rows after removing missing dates and statuses:", len(df))

Rows after removing missing dates and statuses: 990


## Task 1.3 – Clean, Pass 2: Validity, Gaps and Outliers

Objective: Handle invalid ages, impute missing values using medians, detect and cap outliers using the IQR method, and round numerical columns.

In [13]:

import numpy as np

# Step 1: Identify invalid ages

# Identify ages below 1 or above 100
invalid_age = (df["age"] < 1) | (df["age"] > 100)

# Count invalid age values before replacing them
print("Invalid age values:", invalid_age.sum())

# Replace invalid ages with NaN
df.loc[invalid_age, "age"] = np.nan

# Check total missing age values after validation
print("Total missing ages:", df["age"].isnull().sum())


Invalid age values: 5
Total missing ages: 17


In [14]:

# Step 2: Fill missing values using median

# Calculate median age and distance
age_median = df["age"].median()
distance_median = df["distance_km"].median()

# Fill missing values with their respective medians
df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

# Display median values used for imputation
print("Median age:", age_median)
print("Median distance:", distance_median)

# Verify that no missing values remain
print("Missing ages:", df["age"].isnull().sum())
print("Missing distances:", df["distance_km"].isnull().sum())

Median age: 48.0
Median distance: 8.25
Missing ages: 0
Missing distances: 0


In [15]:

# Step 3: Detect and cap outliers using the IQR method

# Process distance and fee columns
for col in ["distance_km", "fee_inr"]:

    # Calculate first and third quartiles
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    # Calculate interquartile range
    IQR = Q3 - Q1

    # Calculate lower and upper outlier boundaries
    lower_fence = Q1 - (1.5 * IQR)
    upper_fence = Q3 + (1.5 * IQR)

    # Count values outside the boundaries before capping
    capped_count = (
        (df[col] < lower_fence) | (df[col] > upper_fence)
    ).sum()

    # Cap outliers within the calculated boundaries
    df[col] = np.clip(df[col], lower_fence, upper_fence)

    # Display outlier capping results
    print(f"\nColumn: {col}")
    print("Q1:", Q1)
    print("Q3:", Q3)
    print("IQR:", IQR)
    print("Lower fence:", lower_fence)
    print("Upper fence:", upper_fence)
    print("Values capped:", capped_count)



Column: distance_km
Q1: 4.625
Q3: 13.0
IQR: 8.375
Lower fence: -7.9375
Upper fence: 25.5625
Values capped: 26

Column: fee_inr
Q1: 500.0
Q3: 787.5
IQR: 287.5
Lower fence: 68.75
Upper fence: 1218.75
Values capped: 3


In [14]:

# Step 4: Round numerical columns

# Round distance to one decimal place
df["distance_km"] = df["distance_km"].round(1)

# Round fee to whole rupees and convert to integer
df["fee_inr"] = df["fee_inr"].round().astype(int)

# Display the cleaned numerical columns
print(df[["age", "distance_km", "fee_inr"]].head())

# Display final dataset shape
print("\nFinal dataset shape:", df.shape)


   age  distance_km  fee_inr
0   60         18.0      850
1   39         12.5      350
2   63          4.1      850
4   52          7.5      750
5   64         19.0      700

Final dataset shape: (990, 14)


In [16]:
# Final verification for Task 1.3

print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df[["age", "distance_km", "fee_inr"]].isnull().sum())

print("\nMaximum fee:", df["fee_inr"].max())
print("Maximum distance:", df["distance_km"].max())

print("\nData types:")
print(df[["age", "distance_km", "fee_inr"]].dtypes)

Dataset shape: (990, 14)

Missing values:
age            0
distance_km    0
fee_inr        0
dtype: int64

Maximum fee: 1218.75
Maximum distance: 25.5625

Data types:
age              int64
distance_km    float64
fee_inr        float64
dtype: object



## Task 1.4 – Features and a NumPy Summary

**Objective:** Create derived features from appointment dates and status, then use NumPy arrays and vectorized operations to calculate summary statistics, booking lead-time counts, and distance z-scores.


In [17]:

# Step 1: Create lead_days and no_show features

# Calculate the number of days between booking and appointment
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Convert appointment status into a binary no-show indicator
# 1 = No-Show, 0 = Attended
df["no_show"] = (df["status"] == "No-Show").astype(int)

# Display the newly created features
print(df[["booked_on", "appt_date", "lead_days", "status", "no_show"]].head())

# Verify the feature values
print("\nLead days range:", df["lead_days"].min(), "to", df["lead_days"].max())
print("Total no-shows:", df["no_show"].sum())


   booked_on  appt_date  lead_days    status  no_show
0 2025-01-24 2025-01-25          1   No-Show        1
1 2025-05-27 2025-06-06         10  Attended        0
2 2025-03-24 2025-03-30          6  Attended        0
4 2025-04-18 2025-04-22          4  Attended        0
5 2024-11-23 2025-01-02         40   No-Show        1

Lead days range: 1 to 45
Total no-shows: 218


In [18]:
# Step 1.1: Calculate appointments booked more than 14 days ahead

long_lead_count = (df["lead_days"] > 14).sum()

long_lead_percentage = (long_lead_count / len(df)) * 100

# Display the results
print("Appointments with lead days > 14:", long_lead_count)
print("Percentage of appointments:", round(long_lead_percentage, 1), "%")

Appointments with lead days > 14: 296
Percentage of appointments: 29.9 %


In [17]:

# Step 2: Convert selected columns into a NumPy array

# Select numerical features for statistical analysis
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# Calculate the mean of each column using NumPy
column_means = np.mean(X, axis=0)

# Display column names and their corresponding means
print("Column means:")
print("Age:", column_means[0])
print("Distance (km):", column_means[1])
print("Fee (INR):", column_means[2])
print("Lead days:", column_means[3])

# Verify the shape of the NumPy array
print("\nNumPy array shape:", X.shape)


Column means:
Age: 48.173737373737374
Distance (km): 9.50090909090909
Fee (INR): 635.7141414141414
Lead days: 11.819191919191919

NumPy array shape: (990, 4)


In [19]:

# Step 3: Calculate z-scores for distance_km

# Convert distance column into a NumPy array
x = df["distance_km"].to_numpy()

# Calculate z-scores using NumPy population standard deviation
z = (x - np.mean(x)) / np.std(x)

# Count records where absolute z-score is greater than 2
extreme_count = np.sum(np.abs(z) > 2)

# Display the results
print("Mean distance:", np.mean(x))
print("Standard deviation:", np.std(x))
print("Records with |z| > 2:", extreme_count)


Mean distance: 9.500934343434343
Standard deviation: 6.154420329663393
Records with |z| > 2: 53



## Task 1.5 – Enrich the Clinics from a Live API 

**Objective:** Fetch temperature data for all 8 clinics using the Open-Meteo API, handle API failures with one retry, and export the enriched clinic dataset as clinics_enriched.csv.


In [20]:

# Step 1: Load the original clinic dataset

import requests

clinics = pd.read_csv("data/clinics.csv")

# Inspect clinic records and column names
print("Clinic dataset shape:", clinics.shape)
print("\nClinic columns:")
print(clinics.columns.tolist())

print("\nClinic records:")
print(clinics.head(8))


Clinic dataset shape: (8, 6)

Clinic columns:
['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']

Clinic records:
  clinic_id               clinic_name     city    tier  latitude  longitude
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462


In [21]:

# Step 2: Fetch temperature data using Open-Meteo API

# Define the Open-Meteo forecast API endpoint
api_url = "https://api.open-meteo.com/v1/forecast"

# Store temperature readings and API response status codes
temperatures = []
api_status_codes = []

# Iterate through each clinic
for _, clinic in clinics.iterrows():

    # Prepare API parameters using clinic coordinates
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    # Initialize temperature, failure reason and status code
    temp_c = None
    failure_reason = None
    status_code = None

    # Attempt the API request twice if necessary
    for attempt in range(2):
        try:
            response = requests.get(
                api_url,
                params=params,
                timeout=10
            )

            # Capture the actual HTTP response status
            status_code = response.status_code

            # Check whether the API request was successful
            if response.status_code == 200:

                # Read the API response as JSON
                weather_data = response.json()

                # Extract the first hourly temperature reading
                hourly_data = weather_data.get("hourly", {})
                temp_values = hourly_data.get("temperature_2m", [])

                if temp_values:
                    temp_c = temp_values[0]
                    break
                else:
                    failure_reason = "Hourly temperature data missing"

            else:
                failure_reason = f"HTTP status {response.status_code}"

        except requests.exceptions.RequestException as error:
            failure_reason = str(error)

    # Report failure if both attempts were unsuccessful
    if temp_c is None:
        print(
            f"API failed for {clinic['clinic_name']}: "
            f"{failure_reason}"
        )

    # Store the temperature and final response status
    temperatures.append(temp_c)
    api_status_codes.append(status_code)

# Add temperature readings to the clinic dataset
clinics["temp_c"] = temperatures

# Display the enriched clinic dataset
print("\nEnriched clinic data:")
print(clinics[["clinic_name", "city", "temp_c"]])

# Step 2.1: Verify API response status codes

# Count successful and failed HTTP responses
successful_responses = api_status_codes.count(200)

failed_responses = sum(
    code is not None and code != 200
    for code in api_status_codes
)

missing_status_codes = api_status_codes.count(None)

# Display API verification results
print("\nAPI Response Verification:")
print("Total API calls:", len(api_status_codes))
print("HTTP 200 responses:", successful_responses)
print("Failed API responses:", failed_responses)
print("Missing status codes:", missing_status_codes)



Enriched clinic data:
                clinic_name     city  temp_c
0     Kothrud Family Clinic     Pune    17.6
1      Hadapsar Care Centre     Pune    19.0
2    Sitabuldi Health Point   Nagpur    20.4
3      Malviya Nagar Clinic   Jaipur    18.6
4     Vaishali Nagar Clinic   Jaipur    18.6
5  Edappally Medical Centre    Kochi    26.3
6        Vijay Nagar Clinic   Indore    20.0
7    Gomti Nagar Care Point  Lucknow    19.2

API Response Verification:
Total API calls: 8
HTTP 200 responses: 8
Failed API responses: 0
Missing status codes: 0


In [22]:

# Step 3: Export enriched clinic dataset

# Save the enriched clinic data as a CSV file
clinics.to_csv("clinics_enriched.csv", index=False)

# Verify the exported dataset
print("clinics_enriched.csv created successfully!")

print("\nDataset shape:", clinics.shape)
print("Total clinics:", len(clinics))

# Check for missing temperature values
print("Missing temperature values:", clinics["temp_c"].isnull().sum())

# Display the final dataset
print("\nFinal enriched clinic dataset:")
print(clinics.head(8))


clinics_enriched.csv created successfully!

Dataset shape: (8, 7)
Total clinics: 8
Missing temperature values: 0

Final enriched clinic dataset:
  clinic_id               clinic_name     city    tier  latitude  longitude  \
0       C01     Kothrud Family Clinic     Pune   Metro   18.5074    73.8077   
1       C02      Hadapsar Care Centre     Pune   Metro   18.5089    73.9260   
2       C03    Sitabuldi Health Point   Nagpur  Tier-2   21.1458    79.0882   
3       C04      Malviya Nagar Clinic   Jaipur  Tier-2   26.8553    75.8150   
4       C05     Vaishali Nagar Clinic   Jaipur  Tier-2   26.9124    75.7873   
5       C06  Edappally Medical Centre    Kochi  Tier-2   10.0261    76.3085   
6       C07        Vijay Nagar Clinic   Indore  Tier-2   22.7533    75.8937   
7       C08    Gomti Nagar Care Point  Lucknow  Tier-2   26.8467    80.9462   

   temp_c  
0    17.6  
1    19.0  
2    20.4  
3    18.6  
4    18.6  
5    26.3  
6    20.0  
7    19.2  


In [23]:
# Verify API response status codes
# Expected: 8 successful responses with status 200

print("Total clinics:", len(clinics))
print("Temperature values:", clinics["temp_c"].notnull().sum())
print("Missing temperatures:", clinics["temp_c"].isnull().sum())

Total clinics: 8
Temperature values: 8
Missing temperatures: 0



### Task 1.6 – Export Clean Data and Generate SQL Load Script

**Objective:** Validate the cleaned appointment dataset, export exactly 16 required columns in the specified order, and generate SQL INSERT statements for clinics and appointments using Python f-strings.


In [24]:

# Step 1: Define the exact 16 columns required for export

required_columns = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

# Check whether all required columns exist
missing_columns = [
    col for col in required_columns if col not in df.columns
]

print("Missing required columns:", missing_columns)

# Check appointment ID uniqueness
print(
    "Appointment IDs are unique:",
    df["appointment_id"].is_unique
)

# Check missing values in the required columns
missing_values = df[required_columns].isnull().sum()

print("\nMissing values in required columns:")
print(missing_values)

print(
    "\nTotal missing values:",
    missing_values.sum()
)


Missing required columns: []
Appointment IDs are unique: True

Missing values in required columns:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0


In [28]:

# Step 2: Prepare the final appointment dataset

# Select only the required 16 columns
appointments_clean = df[required_columns].copy()

# Sort records by appointment_id
appointments_clean = appointments_clean.sort_values(
    by="appointment_id"
).reset_index(drop=True)

# Convert date columns into YYYY-MM-DD text format
appointments_clean["booked_on"] = (
    pd.to_datetime(appointments_clean["booked_on"])
    .dt.strftime("%Y-%m-%d")
)

appointments_clean["appt_date"] = (
    pd.to_datetime(appointments_clean["appt_date"])
    .dt.strftime("%Y-%m-%d")
)

# Export the cleaned dataset
appointments_clean.to_csv(
    "appointments_clean.csv",
    index=False
)

# Verify the exported dataset
print("appointments_clean.csv created successfully!")
print("Final shape:", appointments_clean.shape)
print("Total columns:", len(appointments_clean.columns))

print("\nFirst five records:")
print(appointments_clean.head())


appointments_clean.csv created successfully!
Final shape: (990, 16)
Total columns: 16

First five records:
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0    850.0           Phone  Consultation           Yes   
1         12.5    350.0         Walk-In     Follow-up            No   
2          4.1    850.0         Walk-In  Consultation            No   
3          7.5    750.0             App  Consultation           Yes   
4         19.0    700.0           Phone  Consultation            No   

   

In [27]:

# Step 3: Generate SQL load script

# Load the exported datasets
clinics_sql = pd.read_csv("clinics_enriched.csv")
appointments_sql = pd.read_csv("appointments_clean.csv")

# Define numerical columns for both tables
clinic_numeric_columns = [
    "latitude",
    "longitude",
    "temp_c"
]

appointment_numeric_columns = [
    "age",
    "distance_km",
    "fee_inr",
    "prior_no_shows",
    "lead_days",
    "no_show"
]

# Function to format SQL values
def format_sql_value(value, column, numeric_columns):

    # Handle missing values as SQL NULL
    if pd.isna(value):
        return "NULL"

    # Keep numeric values unquoted
    if column in numeric_columns:
        return f"{value:.15g}" if isinstance(value, float) else str(value)

    # Keep text values inside single quotes
    return f"'{value}'"


# Create an empty list for SQL statements
sql_statements = []

# Insert clinic records first
for _, row in clinics_sql.iterrows():

    values = [
        format_sql_value(row[col], col, clinic_numeric_columns)
        for col in clinics_sql.columns
    ]

    sql_statement = (
        f"INSERT INTO clinics VALUES ({','.join(values)});"
    )

    sql_statements.append(sql_statement)


# Insert appointment records after clinics
for _, row in appointments_sql.iterrows():

    values = [
        format_sql_value(row[col], col, appointment_numeric_columns)
        for col in appointments_sql.columns
    ]

    sql_statement = (
        f"INSERT INTO appointments VALUES ({','.join(values)});"
    )

    sql_statements.append(sql_statement)


# Write all SQL statements into load_data.sql
with open("load_data.sql", "w", encoding="utf-8") as file:
    for statement in sql_statements:
        file.write(statement + "\n")


# Display verification results
print("load_data.sql created successfully!")
print("Clinic INSERT statements:", len(clinics_sql))
print("Appointment INSERT statements:", len(appointments_sql))
print("Total SQL statements:", len(sql_statements))

# Display sample SQL statements
print("\nFirst clinic INSERT:")
print(sql_statements[0])

print("\nFirst appointment INSERT:")
print(sql_statements[len(clinics_sql)])


load_data.sql created successfully!
Clinic INSERT statements: 8
Appointment INSERT statements: 990
Total SQL statements: 998

First clinic INSERT:
INSERT INTO clinics VALUES ('C01','Kothrud Family Clinic','Pune','Metro',18.5074,73.8077,17.6);

First appointment INSERT:
INSERT INTO appointments VALUES ('A0001','P509','C05','2025-01-24','2025-01-25',60,'F',18,850,'Phone','Consultation','Yes',1,'No-Show',1,1);


In [29]:
# Final acceptance criteria verification

print("Total appointment rows:", len(appointments_clean))

print(
    "Distinct appointment IDs:",
    appointments_clean["appointment_id"].nunique()
)

print(
    "Total missing values:",
    appointments_clean.isnull().sum().sum()
)

print("Total columns:", len(appointments_clean.columns))

print(
    "Appointment IDs unique:",
    appointments_clean["appointment_id"].is_unique
)

Total appointment rows: 990
Distinct appointment IDs: 990
Total missing values: 0
Total columns: 16
Appointment IDs unique: True


In [30]:

# ============================================================
# TASK 2.2: VERIFY SQL RESULTS USING PYTHON GROUPBY
#
# Objective:
# Reproduce clinic-wise appointment counts and no-show
# counts using pandas and compare them with SQL results.
# ============================================================

# Group appointment records by clinic
clinic_summary = (
    df.groupby("clinic_id")
    .agg(
        appointments=("appointment_id", "count"),
        no_shows=("no_show", "sum")
    )
    .reset_index()
)

# Add clinic names and city details
clinic_summary = clinic_summary.merge(
    clinics[["clinic_id", "clinic_name", "city"]],
    on="clinic_id",
    how="left"
)

# Calculate clinic-wise no-show percentage
clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"]
    / clinic_summary["appointments"]
).round(1)

# Sort clinics by no-show percentage (highest first)
clinic_summary = clinic_summary.sort_values(
    by="no_show_pct",
    ascending=False
)

# Display the final clinic-wise summary
print("TASK 2.2: PYTHON GROUPBY VERIFICATION")

print(
    clinic_summary[
        [
            "clinic_name",
            "city",
            "appointments",
            "no_shows",
            "no_show_pct"
        ]
    ].to_string(index=False)
)


TASK 2.2: PYTHON GROUPBY VERIFICATION
             clinic_name    city  appointments  no_shows  no_show_pct
  Gomti Nagar Care Point Lucknow            90        30         33.3
    Hadapsar Care Centre    Pune           143        39         27.3
    Malviya Nagar Clinic  Jaipur           142        38         26.8
   Kothrud Family Clinic    Pune           160        36         22.5
      Vijay Nagar Clinic  Indore           101        21         20.8
Edappally Medical Centre   Kochi           142        23         16.2
  Sitabuldi Health Point  Nagpur            82        13         15.9
   Vaishali Nagar Clinic  Jaipur           130        18         13.8
